# Traceprop-LLM — inline gradient-logging overhead (MLSys headline)

**Claim under test:** logging per-sample gradient provenance *inline* during a LoRA fine-tune costs **<1% wall-clock**, vs the full second training-set pass that post-hoc methods (TRAK / LoGRA) require.

Run on a GPU runtime (Runtime → Change runtime type → GPU; T4 is fine, A100 for Pythia-1B).

This notebook: (1) installs deps, (2) pulls the repo, (3) measures baseline vs instrumented step time on GPT-2 and Pythia, (4) prints a results table.

## 1. Install dependencies

In [ ]:
!pip -q install transformers peft accelerate
# Colab ships an old torchao (0.10) whose PEFT compatibility check raises on import; we don't use it.
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Get the Traceprop code

The `traceprop.llm` module and `exp25` are on the private repo. Paste a GitHub token (a fine-grained read token for `AmitoVrito/Traceprop`) below. If you prefer, upload the repo zip instead and skip this cell.

In [ ]:
import getpass, os
TOKEN = getpass.getpass('GitHub token (leave blank if uploading manually): ').strip()
if TOKEN:
    url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
    !git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
    %cd /content/Traceprop
    !pip -q install -e .
else:
    print('No token given — upload the repo to /content/Traceprop, then run: %cd /content/Traceprop and !pip install -e .')

In [ ]:
# sanity: import the inline logger
import sys; sys.path.insert(0, '/content/Traceprop/experiments')
from traceprop.llm import LoRAGradientLogger, select_lora_linears
print('traceprop.llm OK')

## 3. Headline: overhead on GPT-2 (124M) + LoRA

Baseline = plain LoRA step. Instrumented = same step + inline per-sample gradient logging. Overhead is the median-step ratio; the store footprint is what you keep for attribution.

In [ ]:
%cd /content/Traceprop/experiments
!python exp25_llm_inline_overhead.py --backend hf --model gpt2 --device cuda \
    --steps 200 --warmup 10 --batch 16 --seq 128 --rank 8 --proj_dim 512

## 4. Scale up: Pythia-410M (and 1B on A100)

In [ ]:
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-410m --device cuda \
    --steps 150 --warmup 10 --batch 8 --seq 128 --rank 8 --proj_dim 512

In [ ]:
# Pythia-1B — fits on L4/A100 (24GB+). Lower batch or skip on T4 (16GB).
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --steps 100 --warmup 10 --batch 4 --seq 128 --rank 8 --proj_dim 512

## 5. Collect results

In [ ]:
import glob, json
rows = [json.load(open(p)) for p in glob.glob('/content/Traceprop/experiments/results/exp25_hf_*.json')]
print(f"{'model':<22}{'base_ms':>9}{'inst_ms':>9}{'overhead%':>11}{'store_mb':>10}{'grad_dim':>10}")
for r in sorted(rows, key=lambda r: r['model']):
    print(f"{r['model']:<22}{r['baseline_median_s']*1e3:>9.2f}{r['instrumented_median_s']*1e3:>9.2f}{r['overhead_pct']:>11.3f}{r['store_mb']:>10.2f}{str(r['per_sample_grad_dim']):>10}")